## Current Power BI reports inventory

This notebook retrieves the current state of reports that still exist in Power BI and writes a snapshot table to the lakehouse.

## Imports

In [ ]:
import msal
import requests
from datetime import datetime, timezone

from notebookutils import mssparkutils
from pyspark.sql import functions as F

## Define parameters

In [ ]:
TABLE_NAME = "pbi_current_reports_inventory"
POWERBI_RESOURCE = "https://analysis.windows.net/powerbi/api"
GROUPS_BASE_URL = "https://api.powerbi.com/v1.0/myorg/admin/groups"
PAGE_SIZE = 5000
WORKSPACE_FILTER = None  # Optional list of workspace ids

## Authenticate

In [ ]:
vaulturl = "https://<your-key-vault-name>.vault.azure.net/"
CLIENT_ID = mssparkutils.credentials.getSecret(vaulturl, "usageclient")
CLIENT_SECRET = mssparkutils.credentials.getSecret(vaulturl, "usagesecret")
TENANT_ID = "<your-tenant-id-guid>"

authority = f"https://login.microsoftonline.com/{TENANT_ID}"
app = msal.ConfidentialClientApplication(
    client_id=CLIENT_ID,
    authority=authority,
    client_credential=CLIENT_SECRET
)

result = app.acquire_token_for_client(scopes=[f"{POWERBI_RESOURCE}/.default"])
if "access_token" not in result:
    raise RuntimeError(f"Failed to acquire token via MSAL: {result}")

headers = {
    "Authorization": f"Bearer {result['access_token']}",
    "Content-Type": "application/json"
}

print("Authentication successful using service principal")


## Helpers

In [ ]:
def fetch_groups_with_reports():
    records = []
    skip = 0

    while True:
        url = (
            f"{GROUPS_BASE_URL}?$top={PAGE_SIZE}&$skip={skip}"
            f"&$expand=reports&$filter=state ne 'Deleted'"
        )
        resp = requests.get(url, headers=headers, timeout=120)
        resp.raise_for_status()
        payload = resp.json()
        batch = payload.get("value", []) or []

        if not batch:
            break

        records.extend(batch)
        print(f"Fetched {len(batch)} workspaces (running total: {len(records)})")

        if len(batch) < PAGE_SIZE:
            break

        skip += PAGE_SIZE

    return records


def flatten_reports(groups_payload):
    snapshot_time = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    rows = []

    for workspace in groups_payload:
        workspace_id = workspace.get("id")
        workspace_name = workspace.get("name")
        workspace_state = workspace.get("state")

        if WORKSPACE_FILTER and workspace_id not in WORKSPACE_FILTER:
            continue

        for report in workspace.get("reports", []) or []:
            rows.append({
                "workspace_id": workspace_id,
                "workspace_name": workspace_name,
                "workspace_state": workspace_state,
                "report_id": report.get("id"),
                "report_name": report.get("name"),
                "dataset_id": report.get("datasetId"),
                "web_url": report.get("webUrl"),
                "embed_url": report.get("embedUrl"),
                "report_type": report.get("reportType"),
                "created_date_time": report.get("createdDateTime"),
                "modified_date_time": report.get("modifiedDateTime"),
                "created_by": report.get("createdBy"),
                "modified_by": report.get("modifiedBy"),
                "app_id": report.get("appId"),
                "is_current_report": True,
                "retrieved_at_utc": snapshot_time
            })

    return rows


## Fetch current reports inventory

In [ ]:
groups_payload = fetch_groups_with_reports()
report_rows = flatten_reports(groups_payload)

print(f"Fetched {len(groups_payload)} workspaces")
print(f"Flattened {len(report_rows)} current reports")

## Write snapshot to lakehouse

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType

schema = StructType([
    StructField("report_id", StringType(), True),
    StructField("report_name", StringType(), True),
    StructField("workspace_id", StringType(), True),
    StructField("workspace_name", StringType(), True),
    StructField("dataset_id", StringType(), True),
    StructField("dataset_name", StringType(), True),
    StructField("web_url", StringType(), True),
    StructField("created_by", StringType(), True),
    StructField("is_from_app", BooleanType(), True)
])

if not report_rows:
    print("No current reports returned. Nothing written.")
else:
    normalized_rows = []

    for row in report_rows:
        normalized_rows.append({
            "report_id": str(row.get("report_id")) if row.get("report_id") is not None else None,
            "report_name": str(row.get("report_name")) if row.get("report_name") is not None else None,
            "workspace_id": str(row.get("workspace_id")) if row.get("workspace_id") is not None else None,
            "workspace_name": str(row.get("workspace_name")) if row.get("workspace_name") is not None else None,
            "dataset_id": str(row.get("dataset_id")) if row.get("dataset_id") is not None else None,
            "dataset_name": str(row.get("dataset_name")) if row.get("dataset_name") is not None else None,
            "web_url": str(row.get("web_url")) if row.get("web_url") is not None else None,
            "created_by": str(row.get("created_by")) if row.get("created_by") is not None else None,
            "is_from_app": bool(row.get("is_from_app")) if row.get("is_from_app") is not None else None
        })

    df = spark.createDataFrame(normalized_rows, schema=schema)
    df = df.withColumn("ingestion_date_utc", F.current_date())

    (df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(TABLE_NAME))

    print(f"Overwrote snapshot table {TABLE_NAME} with {df.count()} rows")